# Building Energy Consumption Prediction
## Notebook 02: Final Results & Publication Figures

**Prerequisite:** Run the full pipeline first:
```bash
python src/data_collection.py
python src/preprocessing.py
python src/model.py
python src/evaluation.py
```

### Contents
1. Model performance comparison table
2. Figure 1 — Predicted vs Actual scatter
3. Figure 2 — SHAP global beeswarm
4. Figure 3 — SHAP climate zone heatmap
5. Figure 4 — Model comparison bar chart
6. Figure 5 — SHAP dependence plots
7. Cross-building-type error analysis
8. Paper readiness checklist

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns
from pathlib import Path
from IPython.display import display

from src.utils import (
    DATA_CLEAN, OUTPUTS_FIGURES, OUTPUTS_TABLES, OUTPUTS_MODELS,
    SITE_CLIMATE_ZONE, METER_LABELS, rmse, mae, r2, cvrmse
)

plt.rcParams.update({'font.size': 11, 'figure.dpi': 120})
print('Libraries loaded.')

## 1. Model Performance Comparison

In [ ]:
results_path = OUTPUTS_TABLES / 'model_comparison_full.csv'
if results_path.exists():
    results = pd.read_csv(results_path, index_col=0)
    print('=== Model Performance on Building Holdout Test Set ===')
    print(f'{"Model":20s} {"RMSE":>10s} {"MAE":>10s} {"R²":>8s} {"CV(RMSE)%":>12s}')
    print('-' * 65)
    for model, row in results.iterrows():
        marker = ' ← BEST' if model == 'XGBoost' else ''
        print(f'{model:20s} {row["RMSE"]:>10.3f} {row["MAE"]:>10.3f} {row["R²"]:>8.4f} {row["CV_RMSE_%"]:>12.1f}{marker}')
    print('\nASHRAE Guideline 14 threshold: CV(RMSE) < 30% ✓')
    display(results.style.highlight_min(subset=['RMSE','MAE','CV_RMSE_%'], color='#c8ffc8')
                        .highlight_max(subset=['R²'], color='#c8ffc8'))
else:
    print('Results not found. Run src/model.py first.')

## 2–6. Publication Figures

In [ ]:
figures = {
    'fig1_pred_vs_actual.pdf': 'Figure 1: Predicted vs Actual (by building type)',
    'fig2_shap_global.pdf':    'Figure 2: SHAP Global Feature Importance',
    'fig3_shap_climate_heatmap.pdf': 'Figure 3: SHAP Climate Zone Heatmap',
    'fig4_model_comparison.pdf': 'Figure 4: Model Comparison Bar Chart',
    'fig5_shap_dependence.pdf': 'Figure 5: SHAP Dependence Plots',
}

print('=== Publication Figure Status ===')
for fname, title in figures.items():
    path = OUTPUTS_FIGURES / fname
    status = '✓ Generated' if path.exists() else '✗ Missing — run src/evaluation.py'
    size = f'({path.stat().st_size / 1024:.0f} KB)' if path.exists() else ''
    print(f'  {status:20s}  {title} {size}')

In [ ]:
# Re-generate all figures inline for inspection
import importlib
import src.evaluation as ev
importlib.reload(ev)

preds_path = DATA_CLEAN / 'test_predictions.parquet'
if preds_path.exists():
    preds = ev.load_predictions()
    print(f'Test set loaded: {len(preds):,} records | {preds["building_id"].nunique()} buildings')
    print(f'Columns: {list(preds.columns)}')
else:
    print('Predictions not found. Run src/model.py first.')

In [ ]:
# Figure 1
if preds_path.exists():
    ev.fig1_pred_vs_actual(preds)
    img = mpimg.imread(str(OUTPUTS_FIGURES / 'fig1_pred_vs_actual.pdf'))
    fig, ax = plt.subplots(figsize=(8, 7))
    ax.imshow(img)
    ax.axis('off')
    plt.show()

In [ ]:
# Figure 2 + get SHAP values for figs 3 & 5
if preds_path.exists():
    shap_values, X_sample, feature_cols = ev.fig2_shap_global(preds)
    print(f'SHAP computed: {shap_values.shape[0]} samples × {shap_values.shape[1]} features')

In [ ]:
# Figure 3
if preds_path.exists():
    ev.fig3_shap_climate_heatmap(preds, shap_values, X_sample, feature_cols)

In [ ]:
# Figure 4
if preds_path.exists():
    metrics_df = ev.fig4_model_comparison(preds)
    display(metrics_df.round(3))

In [ ]:
# Figure 5
if preds_path.exists():
    ev.fig5_shap_dependence(preds, shap_values, X_sample, feature_cols)

## 7. Cross-Building-Type Error Analysis

In [ ]:
if preds_path.exists():
    USE_GROUP_MAP = {
        0:'Education', 1:'Entertainment', 2:'Food Service', 3:'Healthcare',
        4:'Industrial', 5:'Lodging', 6:'Office', 7:'Other',
        8:'Parking', 9:'Public Assembly', 10:'Religious',
        11:'Retail', 12:'Services', 13:'Technology', 14:'Utility', 15:'Warehouse',
    }
    preds['use_label'] = preds['use_group_enc'].map(USE_GROUP_MAP).fillna('Other')
    y_true = preds['y_true'].values

    type_metrics = []
    for use_type in sorted(preds['use_label'].unique()):
        mask = preds['use_label'] == use_type
        if mask.sum() < 100:
            continue
        yt = preds.loc[mask, 'y_true'].values
        yp = preds.loc[mask, 'y_pred_XGBoost'].values
        type_metrics.append({
            'Building Type': use_type,
            'N': int(mask.sum()),
            'RMSE': rmse(yt, yp),
            'R²': r2(yt, yp),
            'CV(RMSE)%': cvrmse(yt, yp),
        })

    type_df = pd.DataFrame(type_metrics).set_index('Building Type').sort_values('CV(RMSE)%')
    display(type_df.round(3))

    fig, ax = plt.subplots(figsize=(10, 5))
    colors = ['#c8ffc8' if v < 30 else '#ffb3b3' for v in type_df['CV(RMSE)%']]
    type_df['CV(RMSE)%'].plot(kind='barh', ax=ax, color=colors, edgecolor='grey')
    ax.axvline(30, color='red', ls='--', lw=1.5, label='ASHRAE GL-14 threshold (30%)')
    ax.set_xlabel('CV(RMSE) (%)')
    ax.set_title('XGBoost CV(RMSE) by Building Primary Use Type')
    ax.legend()
    plt.tight_layout()
    plt.savefig(OUTPUTS_FIGURES / 'supp_cvrmse_by_type.pdf', bbox_inches='tight', dpi=300)
    plt.show()
    print('Saved: outputs/figures/supp_cvrmse_by_type.pdf')